# 07. 빈차 효율 시뮬레이션

같은 차량(TAXI_VEHC_ID)의 연속 운행에서 '이전 하차 → 다음 승차' 간격을 빈차시간으로 정의하고,
빈차시간/거리 분포, 비효율 지역, 수요 핫스팟 재배치 시뮬레이션을 분석한다.

> 청크 경계를 넘는 차량 연결을 위해 각 청크의 '마지막 운행'을 다음 청크로 이월(carry-over)한다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import platform
import warnings
warnings.filterwarnings('ignore')

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 6)
import seaborn as sns
import matplotlib.ticker as mticker

In [ ]:
import gc, psutil, os

def mem_usage(tag=''):
    gb = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    print(f'[MEM {tag}] {gb:.2f} GB')

CHUNK_SIZE = 1_000_000
mem_usage('start')

## 1. 청크 집계: 빈차시간 계산 (차량 carry-over)

In [ ]:
D012_PATH = './DC_TBYXD012.csv'
usecols = ['RIDE_DTIME','ALIGHT_DTIME','TAXI_VEHC_ID','PAY_AMT','RIDE_DIST','VACNTV_DIST',
           'RIDE_A_CD','ALIGHT_A_CD','RIDE_POS_X','RIDE_POS_Y','ALIGHT_POS_X','ALIGHT_POS_Y']
dtypes  = {'RIDE_DTIME': str,'ALIGHT_DTIME': str,'TAXI_VEHC_ID': str,'PAY_AMT':'float64',
           'RIDE_DIST':'float64','VACNTV_DIST':'float64','RIDE_A_CD': str,'ALIGHT_A_CD': str,
           'RIDE_POS_X':'float64','RIDE_POS_Y':'float64','ALIGHT_POS_X':'float64','ALIGHT_POS_Y':'float64'}

vac_samples = []     # 빈차구간 레코드 (vacancy_min, VACNTV_DIST, alight_hour, prev_alight_cd, prev pos)
dong_parts, hr_parts, vacdist_parts = [], [], []
carry = None         # 직전 청크의 차량별 마지막 운행
total = 0
SAMPLE_CAP = 3_000_000

for chunk in pd.read_csv(D012_PATH, usecols=usecols, dtype=dtypes, chunksize=CHUNK_SIZE):
    rd = pd.to_datetime(chunk['RIDE_DTIME'], format='%Y%m%d%H%M%S', errors='coerce')
    ad = pd.to_datetime(chunk['ALIGHT_DTIME'], format='%Y%m%d%H%M%S', errors='coerce')
    m = rd.notna() & ad.notna()
    chunk, rd, ad = chunk[m].copy(), rd[m], ad[m]
    chunk['RIDE_DT'], chunk['ALIGHT_DT'] = rd, ad
    chunk['ride_hour'], chunk['alight_hour'] = rd.dt.hour, ad.dt.hour
    total += len(chunk)

    if carry is not None:
        chunk = pd.concat([carry, chunk], ignore_index=True)
    chunk = chunk.sort_values(['TAXI_VEHC_ID','RIDE_DT'])
    chunk['prev_alight_dt'] = chunk.groupby('TAXI_VEHC_ID')['ALIGHT_DT'].shift(1)
    chunk['prev_alight_cd'] = chunk.groupby('TAXI_VEHC_ID')['ALIGHT_A_CD'].shift(1)
    chunk['prev_ax'] = chunk.groupby('TAXI_VEHC_ID')['ALIGHT_POS_X'].shift(1)
    chunk['prev_ay'] = chunk.groupby('TAXI_VEHC_ID')['ALIGHT_POS_Y'].shift(1)
    chunk['vacancy_min'] = (chunk['RIDE_DT'] - chunk['prev_alight_dt']).dt.total_seconds() / 60

    # 다음 청크로 이월할 마지막 운행
    carry = chunk.groupby('TAXI_VEHC_ID').tail(1).drop(
        columns=['prev_alight_dt','prev_alight_cd','prev_ax','prev_ay','vacancy_min'])

    valid = chunk[(chunk['vacancy_min'] > 0) & (chunk['vacancy_min'] <= 480)]
    if len(valid):
        s = valid[['vacancy_min','VACNTV_DIST','alight_hour','prev_alight_cd','prev_ax','prev_ay']]
        if sum(len(x) for x in vac_samples) < SAMPLE_CAP:
            vac_samples.append(s.copy())
        dong_parts.append(valid.groupby('prev_alight_cd').agg(
            vmin_sum=('vacancy_min','sum'), vdist_sum=('VACNTV_DIST','sum'), cnt=('vacancy_min','size')).reset_index())
        hr_parts.append(valid.groupby('alight_hour').agg(
            vmin_sum=('vacancy_min','sum'), cnt=('vacancy_min','size')).reset_index())
    vd = chunk[chunk['VACNTV_DIST'] > 0]
    if len(vd):
        vacdist_parts.append(vd.groupby('ride_hour').agg(vdist_sum=('VACNTV_DIST','sum'), cnt=('VACNTV_DIST','size')).reset_index())
    del chunk, rd, ad; gc.collect()

vacancy = pd.concat(vac_samples, ignore_index=True)
dong_vac = pd.concat(dong_parts).groupby('prev_alight_cd').sum().reset_index()
dong_vac['avg_vacancy_min'] = dong_vac['vmin_sum'] / dong_vac['cnt']
dong_vac['avg_vacancy_dist'] = dong_vac['vdist_sum'] / dong_vac['cnt']
hr_vac = pd.concat(hr_parts).groupby('alight_hour').sum().reset_index()
hr_vac['avg_vacancy_min'] = hr_vac['vmin_sum'] / hr_vac['cnt']
vacdist_hr = pd.concat(vacdist_parts).groupby('ride_hour').sum().reset_index()
vacdist_hr['avg'] = vacdist_hr['vdist_sum'] / vacdist_hr['cnt']
del vac_samples, dong_parts, hr_parts, vacdist_parts; gc.collect()

print(f"전체 {total:,}건, 빈차 샘플 {len(vacancy):,}건")
print(f"평균 빈차시간 {vacancy['vacancy_min'].mean():.1f}분, 중앙값 {vacancy['vacancy_min'].median():.1f}분")
mem_usage('after load')

## 2. 빈차시간 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].hist(vacancy['vacancy_min'], bins=100, color='#5C6BC0', edgecolor='white', alpha=0.8)
axes[0].axvline(vacancy['vacancy_min'].median(), color='red', ls='--',
                label=f"중앙값 {vacancy['vacancy_min'].median():.0f}분")
axes[0].set_title('빈차시간 분포'); axes[0].set_xlabel('빈차시간(분)'); axes[0].legend()
axes[1].bar(hr_vac['alight_hour'], hr_vac['avg_vacancy_min'], color='#FF7043')
axes[1].set_title('시간대별 평균 빈차시간'); axes[1].set_xlabel('하차 시간대'); axes[1].set_ylabel('평균(분)'); axes[1].set_xticks(range(24))
plt.tight_layout(); plt.show()

## 3. 빈차거리 분포 및 시간대별 평균

In [ ]:
vd = vacancy[vacancy['VACNTV_DIST'] > 0]
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].hist(vd['VACNTV_DIST'], bins=100, color='#26A69A', edgecolor='white', alpha=0.8)
axes[0].axvline(vd['VACNTV_DIST'].median(), color='red', ls='--', label=f"중앙값 {vd['VACNTV_DIST'].median():,.0f}m")
axes[0].set_title('빈차거리 분포'); axes[0].set_xlabel('빈차거리(m)'); axes[0].legend()
axes[1].bar(vacdist_hr['ride_hour'], vacdist_hr['avg'], color='#AB47BC')
axes[1].set_title('시간대별 평균 빈차거리'); axes[1].set_xlabel('승차 시간대'); axes[1].set_ylabel('평균(m)'); axes[1].set_xticks(range(24))
plt.tight_layout(); plt.show()
print(f"평균 빈차거리 {vd['VACNTV_DIST'].mean():,.0f}m, 중앙값 {vd['VACNTV_DIST'].median():,.0f}m")

## 4. 비효율 구간: 빈차시간 상위 10%의 하차 위치

In [ ]:
threshold = vacancy['vacancy_min'].quantile(0.9)
top10pct = vacancy[vacancy['vacancy_min'] >= threshold].copy()
print(f"상위 10% 기준 {threshold:.1f}분, {len(top10pct):,}건")
ineff = top10pct['prev_alight_cd'].value_counts().head(20)
fig, ax = plt.subplots(figsize=(14, 6))
ax.barh(ineff.index.astype(str), ineff.values, color='#EF5350')
ax.set_title(f'빈차 상위10%(>={threshold:.0f}분) 하차 행정동 Top20', fontweight='bold')
ax.set_xlabel('비효율 건수'); ax.invert_yaxis()
plt.tight_layout(); plt.show()

## 5. 행정동별 빈차 효율

In [ ]:
eff = dong_vac[dong_vac['cnt'] >= 30].copy()
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
wt = eff.nlargest(15, 'avg_vacancy_min')
axes[0].barh(wt['prev_alight_cd'].astype(str), wt['avg_vacancy_min'], color='#EF5350')
axes[0].set_title('평균 빈차시간 상위 15'); axes[0].invert_yaxis(); axes[0].set_xlabel('분')
wd = eff.nlargest(15, 'avg_vacancy_dist')
axes[1].barh(wd['prev_alight_cd'].astype(str), wd['avg_vacancy_dist'], color='#FF7043')
axes[1].set_title('평균 빈차거리 상위 15'); axes[1].invert_yaxis(); axes[1].set_xlabel('m')
plt.tight_layout(); plt.show()

## 6. 최적 재배치 시뮬레이션 (수요 핫스팟 이동)

In [ ]:
# 수요 핫스팟: 빈차 샘플의 하차 행정동 빈도 상위 20 (좌표는 prev pos 평균)
top_cd = top10pct['prev_alight_cd'].value_counts().head(20).index
hot = top10pct[top10pct['prev_alight_cd'].isin(top_cd)].groupby('prev_alight_cd').agg(
    lat=('prev_ay','mean'), lon=('prev_ax','mean')).reset_index()
hot['lat_deg'] = hot['lat'] / 1e7
hot['lon_deg'] = hot['lon']  # POS_X는 도 단위로 가정 (원본 로직 유지)

sim = top10pct.dropna(subset=['prev_ax','prev_ay']).copy()
sim['lat'] = sim['prev_ay'] / 1e7
sim['lon'] = sim['prev_ax']
sim = sim[(sim['lat']>33)&(sim['lat']<39)&(sim['lon']>124)&(sim['lon']<132)]

hc = hot[['lat_deg','lon_deg']].values
def nearest_km(lat, lon):
    dlat = (lat - hc[:,0]) * 111
    dlon = (lon - hc[:,1]) * 111 * np.cos(np.radians((lat + hc[:,0])/2))
    return np.sqrt(dlat**2 + dlon**2).min()
sim['hot_km'] = [nearest_km(la, lo) for la, lo in zip(sim['lat'].values, sim['lon'].values)]
sim['move_min'] = sim['hot_km'] / 30 * 60          # 30km/h 가정
sim['saved_min'] = (sim['vacancy_min'] - sim['move_min']).clip(lower=0)

print(f"대상 {len(sim):,}건")
print(f"평균 기존 빈차 {sim['vacancy_min'].mean():.1f}분, 평균 이동 {sim['move_min'].mean():.1f}분, 평균 절감 {sim['saved_min'].mean():.1f}분")
print(f"총 절감 {sim['saved_min'].sum()/60:,.0f}시간, 절감율 {sim['saved_min'].sum()/sim['vacancy_min'].sum()*100:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].hist(sim['vacancy_min'], bins=50, alpha=0.6, color='#EF5350', label='기존 빈차')
axes[0].hist(sim['move_min'], bins=50, alpha=0.6, color='#4CAF50', label='핫스팟 이동')
axes[0].set_title('기존 빈차 vs 이동시간'); axes[0].set_xlabel('분'); axes[0].legend()
axes[1].hist(sim['saved_min'], bins=50, color='#2196F3', edgecolor='white', alpha=0.8)
axes[1].set_title('절감 가능 시간 분포'); axes[1].set_xlabel('분')
plt.tight_layout(); plt.show()

## 7. 시간대 × 행정동 빈차 히트맵

In [ ]:
# 상위 30 행정동 × 하차시간대 (샘플 기반)
top30 = top10pct['prev_alight_cd'].value_counts().head(30).index
hm = vacancy[vacancy['prev_alight_cd'].isin(top30)].pivot_table(
    index='prev_alight_cd', columns='alight_hour', values='vacancy_min', aggfunc='mean', fill_value=0)
fig, ax = plt.subplots(figsize=(18, 10))
sns.heatmap(hm.astype(int), cmap='YlOrRd', ax=ax, linewidths=0.5, cbar_kws={'label':'평균 빈차시간(분)'})
ax.set_title('시간대별 빈차 효율 히트맵 (하차 행정동 상위 30)', fontweight='bold')
ax.set_xlabel('하차 시간대'); ax.set_ylabel('하차 행정동')
plt.tight_layout(); plt.show()

## 8. 요약

In [ ]:
print('=== 빈차 효율 시뮬레이션 요약 ===')
print(f"빈차 샘플 {len(vacancy):,}건")
print(f"평균 빈차시간 {vacancy['vacancy_min'].mean():.1f}분 / 중앙값 {vacancy['vacancy_min'].median():.1f}분")
print(f"평균 빈차거리 {vd['VACNTV_DIST'].mean():,.0f}m")
print(f"재배치 대상 {len(sim):,}건, 총 절감 {sim['saved_min'].sum()/60:,.0f}시간, 절감율 {sim['saved_min'].sum()/sim['vacancy_min'].sum()*100:.1f}%")